In [ ]:
"""
Script per convertire un classificatore XGBoost da joblib a ONNX
e verificare la consistenza delle predizioni tra i due formati.
"""

import numpy as np
import joblib
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType
import onnxruntime as rt
import xgboost as xgb
from onnxconverter_common import FloatTensorType as OnnxFloatTensorType
from onnxmltools import convert_xgboost
import time
from sklearn.metrics import accuracy_score, classification_report
import warnings
warnings.filterwarnings('ignore')


def load_xgboost_model(model_path):
    """Carica il modello XGBoost dal file joblib."""
    print(f"Caricamento modello da {model_path}...")
    model = joblib.load(model_path)
    print(f"Modello caricato: {type(model)}")
    return model


def convert_to_onnx(xgb_model, n_features, output_path="model.onnx"):
    """
    Converte il modello XGBoost in formato ONNX.
    
    Args:
        xgb_model: Modello XGBoost addestrato
        n_features: Numero di feature in input
        output_path: Percorso dove salvare il modello ONNX
    """
    print(f"\nConversione in ONNX con {n_features} features...")
    
    # Definisci il tipo di input per ONNX
    initial_types = [('float_input', FloatTensorType([None, n_features]))]
    
    try:
        # Prova prima con onnxmltools (metodo preferito per XGBoost)
        onnx_model = convert_xgboost(
            xgb_model, 
            initial_types=initial_types,
            target_opset=12
        )
        print("Conversione completata con onnxmltools")
    except Exception as e:
        print(f"Tentativo con skl2onnx: {e}")
        # Fallback su skl2onnx se il primo metodo fallisce
        onnx_model = convert_sklearn(
            xgb_model, 
            initial_types=initial_types,
            target_opset=12
        )
        print("Conversione completata con skl2onnx")
    
    # Salva il modello ONNX
    with open(output_path, "wb") as f:
        f.write(onnx_model.SerializeToString())
    
    print(f"Modello ONNX salvato in {output_path}")
    return onnx_model


def create_onnx_session(onnx_path):
    """Crea una sessione ONNX Runtime per l'inferenza."""
    print(f"\nCreazione sessione ONNX da {onnx_path}...")
    sess = rt.InferenceSession(onnx_path)
    
    # Info sul modello
    input_name = sess.get_inputs()[0].name
    output_name = sess.get_outputs()[0].name
    
    print(f"Input name: {input_name}")
    print(f"Output name: {output_name}")
    print(f"Input shape: {sess.get_inputs()[0].shape}")
    
    return sess, input_name, output_name


def generate_test_data(n_samples=1000, n_features=10, seed=42):
    """Genera dati di test casuali per il confronto."""
    np.random.seed(seed)
    X_test = np.random.randn(n_samples, n_features).astype(np.float32)
    return X_test


def compare_predictions(xgb_model, onnx_sess, input_name, output_name, X_test):
    """
    Confronta le predizioni tra XGBoost originale e ONNX.
    
    Returns:
        dict: Metriche di confronto
    """
    print("\n" + "="*60)
    print("CONFRONTO PREDIZIONI")
    print("="*60)
    
    # Predizioni XGBoost
    print("\nEsecuzione predizioni XGBoost...")
    start_time = time.time()
    xgb_proba = xgb_model.predict_proba(X_test)
    xgb_pred = xgb_model.predict(X_test)
    xgb_time = time.time() - start_time
    
    # Predizioni ONNX
    print("Esecuzione predizioni ONNX...")
    start_time = time.time()
    onnx_output = onnx_sess.run([output_name], {input_name: X_test})[0]
    onnx_time = time.time() - start_time
    
    # Gestisci output ONNX (potrebbe essere già le classi o le probabilità)
    if len(onnx_output.shape) == 1:
        # Output è già le classi predette
        onnx_pred = onnx_output
        # Prova a ottenere le probabilità se disponibili
        try:
            prob_output = onnx_sess.run(None, {input_name: X_test})
            if len(prob_output) > 1:
                onnx_proba = prob_output[1]
            else:
                onnx_proba = None
        except:
            onnx_proba = None
    else:
        # Output sono le probabilità
        onnx_proba = onnx_output
        onnx_pred = np.argmax(onnx_proba, axis=1)
    
    # Calcola metriche di confronto
    results = {
        'n_samples': len(X_test),
        'xgb_time': xgb_time,
        'onnx_time': onnx_time,
        'speedup': xgb_time / onnx_time if onnx_time > 0 else 0
    }
    
    # Confronto classi predette
    pred_match = np.array_equal(xgb_pred, onnx_pred)
    pred_accuracy = accuracy_score(xgb_pred, onnx_pred)
    results['predictions_match'] = pred_match
    results['predictions_agreement'] = pred_accuracy
    
    print(f"\n{'Metrica':<30} {'Valore':>15}")
    print("-" * 45)
    print(f"{'Numero campioni:':<30} {results['n_samples']:>15,}")
    print(f"{'Tempo XGBoost (s):':<30} {results['xgb_time']:>15.6f}")
    print(f"{'Tempo ONNX (s):':<30} {results['onnx_time']:>15.6f}")
    print(f"{'Speedup ONNX:':<30} {results['speedup']:>15.2f}x")
    print(f"{'Predizioni identiche:':<30} {'✓' if pred_match else '✗':>15}")
    print(f"{'Agreement rate:':<30} {pred_accuracy:>15.2%}")
    
    # Confronto probabilità se disponibili
    if onnx_proba is not None and hasattr(xgb_proba, 'shape'):
        max_diff = np.max(np.abs(xgb_proba - onnx_proba))
        mean_diff = np.mean(np.abs(xgb_proba - onnx_proba))
        results['max_prob_diff'] = max_diff
        results['mean_prob_diff'] = mean_diff
        
        print(f"{'Differenza max probabilità:':<30} {max_diff:>15.6e}")
        print(f"{'Differenza media probabilità:':<30} {mean_diff:>15.6e}")
        
        # Verifica se le differenze sono trascurabili
        tolerance = 1e-5
        proba_match = max_diff < tolerance
        results['probabilities_match'] = proba_match
        print(f"{'Probabilità equivalenti:':<30} {'✓' if proba_match else '✗':>15}")
        print(f"{'(tolleranza: {})'.format(tolerance):<30} {' ':>15}")
    
    # Mostra alcuni esempi di predizioni
    print("\n" + "="*60)
    print("ESEMPI DI PREDIZIONI (primi 10 campioni)")
    print("="*60)
    print(f"\n{'Indice':<10} {'XGBoost':<15} {'ONNX':<15} {'Match':<10}")
    print("-" * 50)
    for i in range(min(10, len(X_test))):
        match = '✓' if xgb_pred[i] == onnx_pred[i] else '✗'
        print(f"{i:<10} {int(xgb_pred[i]):<15} {int(onnx_pred[i]):<15} {match:<10}")
    
    # Se ci sono differenze, mostra dove
    if not pred_match:
        diff_indices = np.where(xgb_pred != onnx_pred)[0]
        print(f"\n⚠️  Trovate {len(diff_indices)} predizioni differenti su {len(X_test)}")
        print(f"Indici con differenze (primi 20): {diff_indices[:20].tolist()}")
    
    return results


def test_edge_cases(xgb_model, onnx_sess, input_name, output_name, n_features):
    """Testa casi limite per verificare la robustezza della conversione."""
    print("\n" + "="*60)
    print("TEST CASI LIMITE")
    print("="*60)
    
    test_cases = {
        'Valori zero': np.zeros((5, n_features), dtype=np.float32),
        'Valori molto grandi': np.full((5, n_features), 1e6, dtype=np.float32),
        'Valori molto piccoli': np.full((5, n_features), 1e-6, dtype=np.float32),
        'Valori negativi': np.full((5, n_features), -1, dtype=np.float32),
        'Mix di valori': np.array([[0, 1e6, -1e6, 1e-6, -1e-6] + 
                                  [0]*(n_features-5)] * 5, dtype=np.float32)[:, :n_features]
    }
    
    for case_name, test_data in test_cases.items():
        print(f"\n{case_name}:")
        try:
            xgb_pred = xgb_model.predict(test_data)
            onnx_pred = onnx_sess.run([output_name], {input_name: test_data})[0]
            
            if len(onnx_pred.shape) > 1:
                onnx_pred = np.argmax(onnx_pred, axis=1)
            
            match = np.array_equal(xgb_pred, onnx_pred)
            print(f"  XGBoost: {xgb_pred.tolist()}")
            print(f"  ONNX:    {onnx_pred.tolist()}")
            print(f"  Match: {'✓' if match else '✗'}")
        except Exception as e:
            print(f"  ⚠️  Errore: {e}")


def main():
    """Funzione principale per eseguire la conversione e i test."""
    
    # Configurazione
    MODEL_PATH = "model.joblib"  # Modifica con il percorso del tuo modello
    ONNX_PATH = "model.onnx"
    N_FEATURES = 10  # Modifica con il numero di feature del tuo modello
    N_TEST_SAMPLES = 1000
    
    print("="*60)
    print("CONVERSIONE XGBOOST -> ONNX")
    print("="*60)
    
    try:
        # 1. Carica il modello XGBoost
        xgb_model = load_xgboost_model(MODEL_PATH)
        
        # 2. Converti in ONNX
        onnx_model = convert_to_onnx(xgb_model, N_FEATURES, ONNX_PATH)
        
        # 3. Crea sessione ONNX
        onnx_sess, input_name, output_name = create_onnx_session(ONNX_PATH)
        
        # 4. Genera dati di test
        X_test = generate_test_data(N_TEST_SAMPLES, N_FEATURES)
        
        # 5. Confronta le predizioni
        results = compare_predictions(xgb_model, onnx_sess, input_name, output_name, X_test)
        
        # 6. Test casi limite
        test_edge_cases(xgb_model, onnx_sess, input_name, output_name, N_FEATURES)
        
        # 7. Riepilogo finale
        print("\n" + "="*60)
        print("RIEPILOGO CONVERSIONE")
        print("="*60)
        print(f"\n✅ Conversione completata con successo!")
        print(f"📁 Modello ONNX salvato in: {ONNX_PATH}")
        print(f"⚡ Speedup ottenuto: {results['speedup']:.2f}x")
        
        if results.get('predictions_match', False):
            print(f"✅ Le predizioni sono identiche tra XGBoost e ONNX")
        elif results.get('predictions_agreement', 0) > 0.99:
            print(f"⚠️  Le predizioni sono quasi identiche ({results['predictions_agreement']:.2%} di agreement)")
        else:
            print(f"❌ Attenzione: differenze significative nelle predizioni")
            
    except FileNotFoundError:
        print(f"\n❌ Errore: File {MODEL_PATH} non trovato.")
        print("   Assicurati di modificare MODEL_PATH con il percorso corretto del tuo modello.")
    except Exception as e:
        print(f"\n❌ Errore durante la conversione: {e}")
        print("   Verifica che il modello sia un classificatore XGBoost valido.")


if __name__ == "__main__":
    main()